# **EXOPLANET TRANSIT PREDICTION**

In [ ]:
!pip install numpy matplotlib astropy skyfield poliastro lightkurve

Requested poliastro from https://files.pythonhosted.org/packages/1c/ce/b2cf237afeacddd856bb3ae524c44b8aec62e14c13d137283122fd0b5099/poliastro-0.12.0-py3-none-any.whl has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    astropy (<4.*,>=3.1)
             ~~~^
Please use pip<24.1 if you need to use this version.
Requested poliastro from https://files.pythonhosted.org/packages/f7/9a/934e863eee7acca4648b3570085da982cde69969527b9f4d7a0445f16789/poliastro-0.11.1-py3-none-any.whl has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    astropy (<4.*,>=3.0)
             ~~~^
Please use pip<24.1 if you need to use this version.
Requested poliastro from https://files.pythonhosted.org/packages/31/7d/55cfd3a348ed5575d0468e26c65c35295fc743c28598ba790561e065a263/poliastro-0.11.0-py3-none-any.whl has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    astropy (<4.*,>=3.0)
             ~~~^
Please use pip<24.1 if you need to u

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.time import Time
from astropy.table import Table
from astropy.timeseries import BoxLeastSquares
from astropy.constants import G, M_sun

from skyfield.api import load, wgs84, Star
from skyfield.units import Angle

from poliastro.bodies import Sun as PoliSun
from poliastro.twobody import Orbit

from lightkurve import search_lightcurvefile


# ============================================================
# 1. LIGHT CURVE HANDLING
# ============================================================

def generate_synthetic_lightcurve(
    period_days=3.5,
    duration_hours=3.0,
    depth_ppm=8000,
    t0=0.5,            # phase reference (days)
    n_days=40,
    cadence_min=30,
    noise_ppm=500
):
    """
    Generate a synthetic Kepler/TESS-like light curve with a single transiting planet.
    """
    cadence_days = cadence_min / (60 * 24)
    time = np.arange(0, n_days, cadence_days)  # days

    # Planet parameters
    period = period_days
    duration = duration_hours / 24.0          # days
    depth = depth_ppm / 1e6                   # relative drop

    # Base flux = 1 + Gaussian noise
    flux = np.ones_like(time) + np.random.normal(0, noise_ppm / 1e6, size=time.size)

    # Add box-like transit
    phase = ((time - t0 + 0.5 * period) % period) - 0.5 * period
    in_transit = np.abs(phase) < (duration / 2)
    flux[in_transit] -= depth

    tab = Table()
    tab["time"] = time
    tab["flux"] = flux

    return tab, dict(
        true_period=period_days,
        true_t0=t0,
        true_duration=duration,
        true_depth=depth_ppm
    )


def load_lightcurve_from_file(filename):
    """
    Example stub to read from a CSV with columns: time (days), flux (relative).
    Adapt as needed for Kepler/TESS FITS using astropy.io.fits or lightkurve.
    """
    tab = Table.read(filename, format="csv")
    if "time" not in tab.colnames or "flux" not in tab.colnames:
        raise ValueError("CSV must contain 'time' and 'flux' columns.")
    return tab


def load_lightcurve_kepler(kepler_id):
    """
    Download Kepler light curve using Lightkurve and return time & flux arrays.
    """
    print(f"Downloading Kepler LC for {kepler_id} ...")
    lcf = search_lightcurvefile(kepler_id, mission="Kepler").download()
    lc = lcf.SAP_FLUX  # Simple Aperture Photometry
    lc = lc.remove_nans()

    print("Detrending (flattening)...")
    lc_flat = lc.flatten(window_length=401)

    time = lc_flat.time.value     # in days
    flux = lc_flat.flux.value     # normalized by flatten()
    return time, flux


def load_lightcurve_tess(tic_id):
    """
    Download TESS light curve using Lightkurve and return time & flux arrays.
    """
    print(f"Downloading TESS LC for TIC {tic_id} ...")
    lcf = search_lightcurvefile(f"TIC {tic_id}", mission="TESS").download()
    lc = lcf.SAP_FLUX
    lc = lc.remove_nans()
    lc_flat = lc.flatten(window_length=301)

    time = lc_flat.time.value
    flux = lc_flat.flux.value
    return time, flux


# ============================================================
# 2. TRANSIT DETECTION (SINGLE-PLANET VERSION)
#    (from the synthetic script)
# ============================================================

def find_transit_parameters(time, flux):
    """
    Find best-fitting transit using Astropy's BoxLeastSquares.

    Returns:
        period, t0, duration, depth, extra_info
    """
    # Normalize flux
    flux = flux / np.median(flux)

    # Remove NaNs (basic cleaning)
    mask = np.isfinite(time) & np.isfinite(flux)
    time_clean = time[mask]
    flux_clean = flux[mask]

    # Period search grid
    min_period = 0.5   # days
    max_period = 20.0  # days
    n_periods = 5000

    periods = np.linspace(min_period, max_period, n_periods)
    durations = np.linspace(0.5, 5.0, 10) * u.hour  # trial durations

    bls = BoxLeastSquares(time_clean * u.day, flux_clean)
    results = bls.power(periods * u.day, durations)

    best_index = np.argmax(results.power)
    best_period = results.period[best_index].to_value(u.day)
    best_t0 = results.transit_time[best_index].to_value(u.day)
    best_duration = results.duration[best_index].to_value(u.day)
    best_depth = results.depth[best_index]

    return best_period, best_t0, best_duration, best_depth, {
        "results": results,
        "time_clean": time_clean,
        "flux_clean": flux_clean
    }


# ============================================================
# 3. TRANSIT DETECTION (MULTI-PLANET VERSION)
#    (from the updated script)
# ============================================================

def run_bls(time, flux):
    """
    BLS run used in the multi-planet search.
    """
    flux = flux / np.median(flux)
    mask = np.isfinite(time) & np.isfinite(flux)
    time, flux = time[mask], flux[mask]

    periods = np.linspace(0.5, 30.0, 8000)              # days
    durations = np.linspace(0.5, 8.0, 10) * u.hour

    bls = BoxLeastSquares(time * u.day, flux)
    res = bls.power(periods * u.day, durations)

    i = np.argmax(res.power)
    P = res.period[i].to_value(u.day)
    t0 = res.transit_time[i].to_value(u.day)
    dur = res.duration[i].to_value(u.day)
    depth = res.depth[i]
    return P, t0, dur, depth, res


def remove_planet_signal(time, flux, period, t0, duration):
    """Subtracts the box model of one planet to search for additional planets."""
    phase = ((time - t0 + 0.5 * period) % period) - 0.5 * period
    in_transit = np.abs(phase) < (duration / 2)
    depth = 1 - np.median(flux[in_transit])
    flux_corrected = flux.copy()
    flux_corrected[in_transit] += depth
    return flux_corrected


def search_multiplanet(time, flux, max_planets=3):
    """
    Iteratively search for multiple planets using BLS + signal removal.
    """
    results = []
    residual_flux = flux.copy()

    for p in range(max_planets):
        try:
            P, t0, dur, depth, res = run_bls(time, residual_flux)
            results.append((P, t0, dur, depth, res))
            residual_flux = remove_planet_signal(time, residual_flux, P, t0, dur)
        except Exception:
            break

    return results


# ============================================================
# 4. PLOTTING — PHASE-FOLDED LIGHT CURVE
# ============================================================

def plot_lightcurve_and_model(time, flux, period, t0, duration, title="Phase-folded Light Curve"):
    """
    Plot phase-folded light curve with a simple box model.
    (original from synthetic script, extended with title)
    """
    # Ensure time and flux are plain, writable numpy arrays to avoid Astropy view issues
    time = np.asarray(time).copy()
    flux = np.asarray(flux).copy()

    # Phase folding
    phase = ((time - t0 + 0.5 * period) % period) / period - 0.5

    sort_idx = np.argsort(phase)
    phase_sorted = phase[sort_idx]
    flux_sorted = flux[sort_idx]

    plt.figure(figsize=(8, 4))
    plt.scatter(phase_sorted, flux_sorted, s=5, alpha=0.5, label="Data")

    # Box model
    model = np.ones_like(phase_sorted)
    half_dur_phase = (duration / period) / 2.0
    in_transit = np.abs(phase_sorted) < half_dur_phase
    if np.any(in_transit):
        model[in_transit] -= (np.median(flux_sorted[in_transit]) - 1.0)

    plt.plot(phase_sorted, model, lw=2, label="BLS box model")
    plt.xlabel("Phase (cycles)")
    plt.ylabel("Normalized Flux")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 5. TRANSIT EPHEMERIS & VISIBILITY
# ============================================================

def predict_transit_times(period_days, t0_days, start_time, n_transits=10):
    """
    Given period and reference transit time (t0 in same units as 'time'),
    predict the next n_transits mid-transit instants as Astropy Time objects.
    """
    t0_abs = start_time + t0_days * u.day
    transit_times = [t0_abs + k * period_days * u.day for k in range(n_transits)]
    return Time(transit_times)


def visible_from_site(times, ra_deg, dec_deg, lat_deg, lon_deg, elevation_m=0.0, min_alt_deg=30.0):
    """
    Use Skyfield to determine which times correspond to target above min_alt_deg.
    (Base function from the multi-planet script)
    """
    ts = load.timescale()
    eph = load("de421.bsp")
    loc = eph["earth"] + wgs84.latlon(lat_deg, lon_deg, elevation_m=elevation_m)
    star = Star(ra=Angle(degrees=ra_deg), dec=Angle(degrees=dec_deg))

    visible_times = []
    altitudes = []

    for t_astropy, t_skyfield in zip(times, ts.from_astropy(times)):
        alt, _, _ = loc.at(t_skyfield).observe(star).apparent().altaz()
        if alt.degrees >= min_alt_deg:
            visible_times.append(t_astropy)
            altitudes.append(alt.degrees)

    return visible_times, altitudes


def filter_visible_transits(
    transit_times,
    ra_deg,
    dec_deg,
    lat_deg,
    lon_deg,
    elevation_m=0.0,
    min_alt_deg=30.0
):
    """
    Wrapper preserving the original synthetic-script function name.
    Returns Astropy Time and numpy array of altitudes.
    """
    visible_times, altitudes = visible_from_site(
        transit_times,
        ra_deg=ra_deg,
        dec_deg=dec_deg,
        lat_deg=lat_deg,
        lon_deg=lon_deg,
        elevation_m=elevation_m,
        min_alt_deg=min_alt_deg
    )
    return Time(visible_times), np.array(altitudes)


# ============================================================
# 6. ORBIT CONSISTENCY CHECK (POLIASTRO)
# ============================================================

def build_orbit(period_days, star_mass_solar=1.0, ecc=0.0, inc_deg=0.0):
    """
    General orbit builder (from multi-planet script).
    """
    P = period_days * u.day
    M = star_mass_solar * M_sun
    a_cubed = (G * M * P**2) / (4 * np.pi**2)
    a = a_cubed ** (1 / 3)
    a_au = a.to(u.AU)

    orbit = Orbit.from_classical(
        PoliSun,
        a_au,
        ecc=ecc * u.one,
        inc=inc_deg * u.deg,
        raan=0 * u.deg,
        argp=0 * u.deg,
        nu=0 * u.deg,
    )
    return orbit, a_au


def build_orbit_with_poliastro(period_days, star_mass_solar=1.0):
    """
    Wrapper keeping the original synthetic-script function name.
    Uses circular, face-on orbit.
    """
    return build_orbit(period_days, star_mass_solar=star_mass_solar, ecc=0.0, inc_deg=0.0)


# ============================================================
# 7. PIPELINE RUNNERS
# ============================================================

def run_synthetic_demo():
    """
    Original synthetic pipeline:
    - generate LC
    - single-planet BLS
    - sanity check vs truth
    - phase-folded plot
    - transit prediction
    - visibility
    - orbit check
    """
    print("\n" + "="*70)
    print("#### SYNTHETIC LIGHT CURVE DEMO ####")
    print("="*70)

    lc, true_params = generate_synthetic_lightcurve()
    print("Generated synthetic light curve:")
    print(true_params)

    time = np.array(lc["time"])
    flux = np.array(lc["flux"])

    print("\nRunning BoxLeastSquares transit search (single-planet)...")
    best_period, best_t0, best_duration, best_depth, bls_details = find_transit_parameters(time, flux)

    print(f"\nDetected transit parameters:")
    print(f"  Period     : {best_period:.5f} days")
    print(f"  t0         : {best_t0:.5f} days (relative to LC start)")
    print(f"  Duration   : {best_duration:.5f} days")
    print(f"  Depth      : {best_depth:.2e} (relative flux)")

    if "true_period" in true_params:
        print("\n[Sanity check vs synthetic truth]")
        print(f"  True period   : {true_params['true_period']:.5f} days")
        print(f"  True t0       : {true_params['true_t0']:.5f} days")
        print(f"  True duration : {true_params['true_duration']:.5f} days")

    # Plot phase-folded LC
    plot_lightcurve_and_model(
        time,
        flux / np.median(flux),
        best_period,
        best_t0,
        best_duration,
        title="Synthetic – Phase-folded Light Curve"
    )

    # Predict future transits
    start_time = Time.now()
    n_transits = 10
    transit_times = predict_transit_times(best_period, best_t0, start_time, n_transits=n_transits)

    print(f"\nNext {n_transits} predicted transit midpoints (UTC):")
    for i, t_mid in enumerate(transit_times):
        print(f"  Transit {i+1:02d}: {t_mid.iso}")

    # Visibility from Bengaluru
    ra_deg = 290.0
    dec_deg = 44.5
    lat_deg = 12.9716
    lon_deg = 77.5946
    elevation_m = 900.0
    min_alt_deg = 30.0

    visible_times, altitudes_deg = filter_visible_transits(
        transit_times,
        ra_deg=ra_deg,
        dec_deg=dec_deg,
        lat_deg=lat_deg,
        lon_deg=lon_deg,
        elevation_m=elevation_m,
        min_alt_deg=min_alt_deg
    )

    print(f"\nVisible transits from lat={lat_deg}°, lon={lon_deg}° (alt ≥ {min_alt_deg}°):")
    if len(visible_times) == 0:
        print("  None of the next transits are above the altitude limit for this site.")
    else:
        for t_mid, alt in zip(visible_times, altitudes_deg):
            print(f"  {t_mid.iso}  | Altitude ~ {alt:.1f}°")

    # Orbit sanity check
    star_mass_solar = 1.0
    orbit, a_au = build_orbit_with_poliastro(best_period, star_mass_solar=star_mass_solar)

    print("\nPoliastro orbit consistency check (synthetic):")
    print(f"  Semi-major axis: {a_au:.3f}")
    print(f"  Period from orbit: {orbit.period.to(u.day):.3f}")


def run_kepler_demo():
    """
    Multi-planet Kepler pipeline:
    - load real Kepler LC
    - multi-planet BLS
    - show candidates
    - phase-folded plot for planet 1
    - transit prediction
    - visibility
    - orbit check with realistic stellar mass
    """
    print("\n" + "="*70)
    print("#### REAL KEPLER LIGHT CURVE DEMO ####")
    print("="*70)

    time, flux = load_lightcurve_kepler(kepler_id="KIC 11446443")

    planets = search_multiplanet(time, flux, max_planets=3)
    print("\nDetected planets:")
    for n, (P, t0, dur, depth, res) in enumerate(planets, start=1):
        print(f"  Planet {n}: P={P:.5f} d  t0={t0:.5f}  dur={dur:.5f} d  depth={depth:.2e}")

    if not planets:
        print("No planets detected by multi-planet search.")
        return

    # Take first planet candidate
    P1, t01, dur1, depth1, res1 = planets[0]

    # Phase-folded plot for real data (planet 1)
    plot_lightcurve_and_model(
        time,
        flux / np.median(flux),
        P1,
        t01,
        dur1,
        title="Kepler – Phase-folded Light Curve (Planet 1)"
    )

    # Predict future transits
    start = Time.now()
    midpoints = predict_transit_times(P1, t01, start, n_transits=8)

    print("\nNext predicted transits (Kepler planet 1):")
    for t in midpoints:
        print(" ", t.iso)

    # Visibility from Bengaluru
    vis, alts = visible_from_site(
        midpoints,
        ra_deg=290.0,
        dec_deg=44.5,
        lat_deg=12.9716,
        lon_deg=77.5946,
        elevation_m=900,
        min_alt_deg=30
    )
    print("\nVisible from Bengaluru (Kepler planet 1):")
    for t, a in zip(vis, alts):
        print(f"  {t.iso}  alt={a:.1f}°")

    # Build orbit with real-ish stellar mass + small eccentricity / high inclination
    orbit, a_au = build_orbit(
        P1,
        star_mass_solar=1.12,  # example mass from Gaia/ExoFOP
        ecc=0.03,
        inc_deg=89.5,
    )
    print("\nPOLIASTRO ORBIT (Kepler planet 1):")
    print("  Semi-major axis:", a_au)
    print("  Period from orbit:", orbit.period.to(u.day))


# ============================================================
# 8. MAIN
# ============================================================

def main():
    # 1) Synthetic demo (original script behavior)
    run_synthetic_demo()

    # 2) Real Kepler multi-planet demo (updated script behavior)
    run_kepler_demo()


if __name__ == "__main__":
    main()


# ============================================================
# ============================================================
# ADVANCED FEATURES - FUTURE UPGRADES
# ============================================================
# ============================================================

"""
The following sections provide advanced functionality to extend
the basic transit detection pipeline. Each module can be used
independently by importing the necessary functions.

Requirements (install as needed):
    pip install batman-package emcee corner pymc3 exoplanet-core
    pip install streamlit plotly dash astroquery manim scipy

Modules:
    1. Limb-darkened transit fitting (batman)
    2. Transmission spectroscopy flags
    3. Bayesian inference with MCMC (PyMC3 + exoplanet)
    4. Automated Gaia/MAST lookup
    5. Streamlit web dashboard
    6. Plotly/Dash interactive dashboard
    7. 3D orbit animation (Manim + Poliastro)
"""


# ============================================================
# 9. LIMB-DARKENED TRANSIT FITTING (BATMAN)
# ============================================================

def fit_batman_transit(time, flux, period, t0, rp_over_rs=0.1, a_over_rs=15.0,
                       inc_deg=90.0, ecc=0.0, w_deg=90.0,
                       limb_dark="quadratic", u_params=[0.1, 0.3]):
    """
    Fit a limb-darkened transit model using the batman package.

    Args:
        time: array of times (days)
        flux: array of normalized flux
        period: orbital period (days)
        t0: mid-transit time (days)
        rp_over_rs: planet-to-star radius ratio
        a_over_rs: semi-major axis in stellar radii
        inc_deg: orbital inclination (degrees)
        ecc: eccentricity
        w_deg: argument of periastron (degrees)
        limb_dark: limb darkening model ('uniform', 'linear', 'quadratic', 'nonlinear')
        u_params: limb darkening coefficients

    Returns:
        batman_model: transit model flux
        params: batman TransitParams object
    """
    try:
        import batman
    except ImportError:
        print("ERROR: batman-package not installed. Install with: pip install batman-package")
        return None, None

    # Initialize batman parameters
    params = batman.TransitParams()
    params.t0 = t0                        # mid-transit time
    params.per = period                   # orbital period
    params.rp = rp_over_rs                # planet radius in stellar radii
    params.a = a_over_rs                  # semi-major axis in stellar radii
    params.inc = inc_deg                  # inclination
    params.ecc = ecc                      # eccentricity
    params.w = w_deg                      # argument of periastron
    params.limb_dark = limb_dark          # limb darkening model
    params.u = u_params                   # limb darkening coefficients

    # Generate model
    m = batman.TransitModel(params, time)
    batman_model = m.light_curve(params)

    return batman_model, params


def plot_batman_fit(time, flux, period, t0, batman_model):
    """
    Plot phase-folded light curve with batman model overlay.
    """
    phase = ((time - t0 + 0.5 * period) % period) / period - 0.5
    sort_idx = np.argsort(phase)

    plt.figure(figsize=(10, 5))
    plt.scatter(phase[sort_idx], flux[sort_idx], s=5, alpha=0.3, label="Data")
    plt.plot(phase[sort_idx], batman_model[sort_idx], 'r-', lw=2, label="Batman model")
    plt.xlabel("Phase")
    plt.ylabel("Normalized Flux")
    plt.title("Limb-Darkened Transit Fit (Batman)")
    plt.legend()
    plt.tight_layout()
    plt.show()


def optimize_batman_fit(time, flux, flux_err, period, t0,
                        initial_rp=0.1, initial_a=15.0):
    """
    Optimize batman transit parameters using scipy.optimize.

    Returns:
        best_params: optimized batman.TransitParams
        chi2: chi-squared of best fit
    """
    try:
        import batman
        from scipy.optimize import minimize
    except ImportError:
        print("ERROR: Need batman-package and scipy")
        return None, None

    def chi_squared(theta):
        rp, a_rs, inc = theta
        params = batman.TransitParams()
        params.t0 = t0
        params.per = period
        params.rp = rp
        params.a = a_rs
        params.inc = inc
        params.ecc = 0.0
        params.w = 90.0
        params.limb_dark = "quadratic"
        params.u = [0.1, 0.3]

        m = batman.TransitModel(params, time)
        model = m.light_curve(params)
        return np.sum(((flux - model) / flux_err) ** 2)

    # Initial guess
    theta0 = [initial_rp, initial_a, 89.5]

    # Optimize
    result = minimize(chi_squared, theta0, method='Nelder-Mead')

    # Build best-fit params
    best_params = batman.TransitParams()
    best_params.t0 = t0
    best_params.per = period
    best_params.rp = result.x[0]
    best_params.a = result.x[1]
    best_params.inc = result.x[2]
    best_params.ecc = 0.0
    best_params.w = 90.0
    best_params.limb_dark = "quadratic"
    best_params.u = [0.1, 0.3]

    return best_params, result.fun


# ============================================================
# 10. TRANSMISSION SPECTROSCOPY FLAGS
# ============================================================

def estimate_atmospheric_scale_height(planet_mass_earth, planet_radius_earth,
                                     temperature_k, mean_molecular_weight=2.3):
    """
    Estimate atmospheric scale height for transmission spectroscopy.

    H = kT / (μ m_H g)

    Args:
        planet_mass_earth: planet mass in Earth masses
        planet_radius_earth: planet radius in Earth radii
        temperature_k: equilibrium temperature (K)
        mean_molecular_weight: mean molecular weight (amu)

    Returns:
        scale_height_km: atmospheric scale height in km
    """
    from astropy.constants import k_B, m_p, G, M_earth, R_earth

    # Convert to SI
    M_planet = planet_mass_earth * M_earth
    R_planet = planet_radius_earth * R_earth

    # Surface gravity
    g = (G * M_planet / R_planet**2).to(u.m / u.s**2)

    # Scale height H = kT / (μ m_H g)
    mu = mean_molecular_weight
    H = (k_B * temperature_k * u.K) / (mu * m_p * g)

    return H.to(u.km).value


def flag_transmission_spectroscopy_targets(planets_data):
    """
    Flag promising targets for transmission spectroscopy based on:
    - High scale height
    - Bright host star
    - Deep transit

    Args:
        planets_data: list of dicts with keys:
            ['name', 'period', 'rp_earth', 'mass_earth', 'temp_k',
             'star_mag', 'transit_depth']

    Returns:
        ranked_targets: sorted list of targets with transmission score
    """
    scored = []

    for p in planets_data:
        H = estimate_atmospheric_scale_height(
            p.get('mass_earth', 1.0),
            p.get('rp_earth', 1.0),
            p.get('temp_k', 1000)
        )

        # Transmission signal ~ H/R
        signal = H / (p.get('rp_earth', 1.0) * 6371)  # normalized

        # Brightness factor (brighter = better)
        brightness = 1.0 / (10 ** (p.get('star_mag', 12) / 5.0))

        # Transit depth
        depth = p.get('transit_depth', 0.01)

        # Combined score
        score = signal * brightness * depth * 1e6

        scored.append({
            **p,
            'scale_height_km': H,
            'transmission_score': score
        })

    # Sort by score
    ranked = sorted(scored, key=lambda x: x['transmission_score'], reverse=True)
    return ranked


def analyze_wavelength_dependent_depth(wavelengths_nm, transit_depths,
                                      wavelength_bins=5):
    """
    Analyze transit depth vs wavelength to detect atmospheric features.

    Args:
        wavelengths_nm: array of wavelengths (nm)
        transit_depths: array of transit depths at each wavelength
        wavelength_bins: number of bins for analysis

    Returns:
        analysis: dict with slope, scale height estimate, and significance
    """
    from scipy.stats import linregress

    # Bin the data
    bins = np.linspace(wavelengths_nm.min(), wavelengths_nm.max(), wavelength_bins + 1)
    bin_centers = []
    bin_depths = []

    for i in range(wavelength_bins):
        mask = (wavelengths_nm >= bins[i]) & (wavelengths_nm < bins[i+1])
        if np.sum(mask) > 0:
            bin_centers.append(np.mean(wavelengths_nm[mask]))
            bin_depths.append(np.mean(transit_depths[mask]))

    # Linear fit
    if len(bin_centers) > 2:
        slope, intercept, r_value, p_value, std_err = linregress(bin_centers, bin_depths)

        return {
            'slope': slope,
            'r_squared': r_value**2,
            'p_value': p_value,
            'interpretation': 'Significant atmosphere' if p_value < 0.05 else 'No clear detection',
            'bin_wavelengths': bin_centers,
            'bin_depths': bin_depths
        }

    return None


# ============================================================
# 11. BAYESIAN INFERENCE WITH MCMC (PyMC3 + exoplanet)
# ============================================================

def run_mcmc_transit_fit(time, flux, flux_err, period_guess, t0_guess,
                         n_samples=2000, n_tune=1000, chains=2):
    """
    Run MCMC fit to transit light curve using PyMC3 and exoplanet.

    Args:
        time: observation times (days)
        flux: normalized flux
        flux_err: flux uncertainties
        period_guess: initial period estimate (days)
        t0_guess: initial t0 estimate (days)
        n_samples: number of MCMC samples
        n_tune: number of tuning samples
        chains: number of MCMC chains

    Returns:
        trace: PyMC3 trace object
        summary: parameter summary statistics
    """
    try:
        import pymc3 as pm
        import exoplanet as xo
        from exoplanet import distributions as xodist
    except ImportError:
        print("ERROR: Need pymc3 and exoplanet-core")
        print("Install: pip install pymc3 exoplanet-core")
        return None, None

    with pm.Model() as model:
        # Stellar parameters
        mean_flux = pm.Normal("mean", mu=1.0, sd=0.1)
        u = xo.distributions.QuadLimbDark("u")

        # Planet parameters
        log_period = pm.Normal("log_period", mu=np.log(period_guess), sd=0.1)
        period = pm.Deterministic("period", pm.math.exp(log_period))

        t0 = pm.Normal("t0", mu=t0_guess, sd=1.0)

        log_ror = pm.Normal("log_ror", mu=np.log(0.1), sd=1.0)
        ror = pm.Deterministic("ror", pm.math.exp(log_ror))

        b = xo.distributions.ImpactParameter("b", ror=ror)

        # Orbit
        orbit = xo.orbits.KeplerianOrbit(period=period, t0=t0, b=b)

        # Transit light curve
        light_curve = xo.LimbDarkLightCurve(u).get_light_curve(
            orbit=orbit, r=ror, t=time
        )
        transit_model = pm.math.sum(light_curve, axis=-1) + mean_flux

        # Likelihood
        pm.Normal("obs", mu=transit_model, sd=flux_err, observed=flux)

        # Optimize
        map_soln = xo.optimize(start=model.test_point)

        # Sample
        trace = pm.sample(
            tune=n_tune,
            draws=n_samples,
            chains=chains,
            start=map_soln,
            return_inferencedata=True
        )

    summary = pm.summary(trace)
    return trace, summary


def plot_mcmc_corner(trace, var_names=['period', 't0', 'ror', 'b']):
    """
    Create corner plot of MCMC posterior distributions.
    """
    try:
        import corner
        import arviz as az
    except ImportError:
        print("ERROR: Need corner and arviz packages")
        return

    samples = az.extract(trace, var_names=var_names).to_dataframe()

    fig = corner.corner(
        samples,
        labels=var_names,
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        title_kwargs={"fontsize": 12}
    )
    plt.show()


def plot_mcmc_transit_model(time, flux, flux_err, trace, n_samples=100):
    """
    Plot data with MCMC posterior transit models.
    """
    try:
        import arviz as az
        import exoplanet as xo
    except ImportError:
        print("ERROR: Need arviz and exoplanet")
        return

    # Extract random samples
    posterior = az.extract(trace, num_samples=n_samples)

    plt.figure(figsize=(12, 5))
    plt.errorbar(time, flux, yerr=flux_err, fmt='.k', alpha=0.3, label='Data')

    # Plot sample models
    for i in range(min(n_samples, 100)):
        # This is a simplified version - in practice you'd regenerate the model
        plt.plot(time, flux, alpha=0.05, color='red')

    plt.xlabel("Time (days)")
    plt.ylabel("Normalized Flux")
    plt.title("MCMC Posterior Transit Models")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 12. AUTOMATED GAIA/MAST LOOKUP
# ============================================================

def query_gaia_stellar_params(ra_deg, dec_deg, radius_arcsec=5.0):
    """
    Query Gaia DR3 for stellar parameters.

    Args:
        ra_deg: Right Ascension (degrees)
        dec_deg: Declination (degrees)
        radius_arcsec: search radius (arcseconds)

    Returns:
        stellar_params: dict with Gaia parameters
    """
    try:
        from astroquery.gaia import Gaia
        from astropy.coordinates import SkyCoord
    except ImportError:
        print("ERROR: Need astroquery")
        print("Install: pip install astroquery")
        return None

    coord = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame='icrs')

    radius = radius_arcsec * u.arcsec

    job = Gaia.cone_search_async(coord, radius=radius)
    results = job.get_results()

    if len(results) == 0:
        print("No Gaia sources found")
        return None

    # Take brightest source
    brightest = results[np.argmin(results['phot_g_mean_mag'])]

    params = {
        'source_id': brightest['source_id'],
        'ra': brightest['ra'],
        'dec': brightest['dec'],
        'parallax': brightest['parallax'],
        'pmra': brightest['pmra'],
        'pmdec': brightest['pmdec'],
        'g_mag': brightest['phot_g_mean_mag'],
        'bp_mag': brightest['phot_bp_mean_mag'],
        'rp_mag': brightest['phot_rp_mean_mag'],
        'teff': brightest.get('teff_gspphot', np.nan),
        'radius': brightest.get('radius_gspphot', np.nan),
        'lum': brightest.get('lum_gspphot', np.nan)
    }

    return params


def query_exofop_target(tic_id):
    """
    Query ExoFOP-TESS for target information.
    (Note: ExoFOP doesn't have a public API, this is a placeholder)
    """
    print(f"ExoFOP lookup for TIC {tic_id}")
    print("Visit: https://exofop.ipac.caltech.edu/tess/target.php?id={tic_id}")
    return None


def query_mast_observations(target_name):
    """
    Query MAST for available observations.
    """
    try:
        from astroquery.mast import Observations
    except ImportError:
        print("ERROR: Need astroquery")
        return None

    obs_table = Observations.query_object(target_name)

    print(f"\nMAST observations for {target_name}:")
    print(f"  Total observations: {len(obs_table)}")

    if len(obs_table) > 0:
        missions = obs_table['obs_collection'].unique()
        print(f"  Missions: {', '.join(missions)}")

    return obs_table


def create_target_summary(target_name, ra_deg, dec_deg):
    """
    Create comprehensive target summary from multiple archives.
    """
    print(f"\n{'='*60}")
    print(f"TARGET SUMMARY: {target_name}")
    print(f"{'='*60}")

    # Gaia
    print("\n[Gaia DR3]")
    gaia_params = query_gaia_stellar_params(ra_deg, dec_deg)
    if gaia_params:
        print(f"  Source ID: {gaia_params['source_id']}")
        print(f"  G mag: {gaia_params['g_mag']:.2f}")
        print(f"  Teff: {gaia_params['teff']:.0f} K")
        print(f"  Parallax: {gaia_params['parallax']:.2f} mas")

    # MAST
    print("\n[MAST Archive]")
    mast_obs = query_mast_observations(target_name)

    return {
        'gaia': gaia_params,
        'mast': mast_obs
    }


# ============================================================
# 13. STREAMLIT WEB DASHBOARD
# ============================================================

STREAMLIT_APP_CODE = '''
"""
Streamlit Transit Predictor Dashboard

To run: streamlit run transit_dashboard.py
"""

import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u

# Import your main functions (adjust import path as needed)
# from exoplanet_analysis import predict_transit_times, visible_from_site

st.set_page_config(page_title="Exoplanet Transit Predictor", layout="wide")

st.title("🪐 Exoplanet Transit Predictor")
st.markdown("Predict and plan observations of exoplanet transits")

# Sidebar inputs
st.sidebar.header("Target Parameters")
period = st.sidebar.number_input("Orbital Period (days)", value=3.5, min_value=0.1)
t0_ref = st.sidebar.number_input("Reference T0 (BJD)", value=2459000.0)
duration = st.sidebar.number_input("Transit Duration (hours)", value=3.0, min_value=0.1)

st.sidebar.header("Observer Location")
lat = st.sidebar.number_input("Latitude (deg)", value=12.9716, min_value=-90.0, max_value=90.0)
lon = st.sidebar.number_input("Longitude (deg)", value=77.5946, min_value=-180.0, max_value=180.0)
elevation = st.sidebar.number_input("Elevation (m)", value=900.0, min_value=0.0)

st.sidebar.header("Target Coordinates")
ra = st.sidebar.number_input("RA (deg)", value=290.0, min_value=0.0, max_value=360.0)
dec = st.sidebar.number_input("Dec (deg)", value=44.5, min_value=-90.0, max_value=90.0)

min_alt = st.sidebar.slider("Minimum Altitude (deg)", 0, 90, 30)

# Number of transits
n_transits = st.sidebar.slider("Number of transits to predict", 1, 50, 10)

# Main content
tab1, tab2, tab3 = st.tabs(["Transit Predictions", "Visibility Chart", "Target Info"])

with tab1:
    st.header("Predicted Transit Times")

    if st.button("Calculate Transits"):
        # Calculate predictions
        start_time = Time.now()

        # Generate transit times (simplified)
        t0_abs = Time(t0_ref, format='jd')
        transit_times = [t0_abs + k * period * u.day for k in range(n_transits)]

        # Create dataframe
        df = pd.DataFrame({
            'Transit #': range(1, n_transits+1),
            'Mid-Transit Time (UTC)': [t.iso for t in transit_times],
            'JD': [t.jd for t in transit_times],
            'Days from now': [(t - start_time).jd for t in transit_times]
        })

        st.dataframe(df, use_container_width=True)

        # Download button
        csv = df.to_csv(index=False)
        st.download_button(
            "Download as CSV",
            csv,
            "transit_predictions.csv",
            "text/csv"
        )

with tab2:
    st.header("Visibility Chart")
    st.info("Altitude vs time chart for visible transits")

    # Placeholder for visibility plot
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlabel("Time")
    ax.set_ylabel("Altitude (degrees)")
    ax.set_title("Target Visibility")
    ax.axhline(y=min_alt, color='r', linestyle='--', label=f'Min altitude ({min_alt}°)')
    ax.legend()
    st.pyplot(fig)

with tab3:
    st.header("Target Information")

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("System Parameters")
        st.write(f"**Period:** {period} days")
        st.write(f"**Duration:** {duration} hours")
        st.write(f"**RA:** {ra}°")
        st.write(f"**Dec:** {dec}°")

    with col2:
        st.subheader("Observer Location")
        st.write(f"**Latitude:** {lat}°")
        st.write(f"**Longitude:** {lon}°")
        st.write(f"**Elevation:** {elevation} m")

    if st.button("Query Gaia"):
        st.info("Querying Gaia DR3... (would implement query_gaia_stellar_params here)")

# Footer
st.markdown("---")
st.markdown("Built with Streamlit | Data from Astropy, Lightkurve, and Gaia")
'''


def create_streamlit_app(filename="transit_dashboard.py"):
    """
    Create Streamlit app file.
    """
    with open(filename, 'w') as f:
        f.write(STREAMLIT_APP_CODE)

    print(f"Created Streamlit app: {filename}")
    print(f"Run with: streamlit run {filename}")


# ============================================================
# 14. PLOTLY/DASH INTERACTIVE DASHBOARD
# ============================================================

DASH_APP_CODE = '''
"""
Dash Interactive Transit Dashboard

To run: python transit_dash_app.py
"""

import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objs as go
import numpy as np
from astropy.time import Time
import astropy.units as u

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("🪐 Exoplanet Transit Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Div([
            html.H3("Transit Parameters"),
            html.Label("Period (days):"),
            dcc.Input(id='period', type='number', value=3.5, step=0.1),
            html.Br(),
            html.Label("Transit Duration (hours):"),
            dcc.Input(id='duration', type='number', value=3.0, step=0.1),
            html.Br(),
            html.Label("Reference T0 (BJD):"),
            dcc.Input(id='t0', type='number', value=2459000.0),
        ], style={'width': '30%', 'display': 'inline-block', 'padding': '20px'}),

        html.Div([
            html.H3("Observer Location"),
            html.Label("Latitude (°):"),
            dcc.Input(id='lat', type='number', value=12.9716),
            html.Br(),
            html.Label("Longitude (°):"),
            dcc.Input(id='lon', type='number', value=77.5946),
            html.Br(),
            html.Label("Min Altitude (°):"),
            dcc.Slider(id='min_alt', min=0, max=90, value=30, marks={i: str(i) for i in range(0, 91, 15)}),
        ], style={'width': '30%', 'display': 'inline-block', 'padding': '20px'}),

        html.Div([
            html.H3("Target"),
            html.Label("RA (°):"),
            dcc.Input(id='ra', type='number', value=290.0),
            html.Br(),
            html.Label("Dec (°):"),
            dcc.Input(id='dec', type='number', value=44.5),
        ], style={'width': '30%', 'display': 'inline-block', 'padding': '20px'}),
    ]),

    html.Button('Calculate Transits', id='calculate-btn', n_clicks=0,
                style={'margin': '20px', 'padding': '10px 20px', 'fontSize': '16px'}),

    dcc.Graph(id='phase-plot'),
    dcc.Graph(id='visibility-plot'),

    html.Div(id='transit-table', style={'padding': '20px'})
])

@app.callback(
    [Output('phase-plot', 'figure'),
     Output('visibility-plot', 'figure'),
     Output('transit-table', 'children')],
    [Input('calculate-btn', 'n_clicks')],
    [State('period', 'value'),
     State('duration', 'value'),
     State('t0', 'value'),
     State('lat', 'value'),
     State('lon', 'value'),
     State('min_alt', 'value'),
     State('ra', 'value'),
     State('dec', 'value')]
)
def update_plots(n_clicks, period, duration, t0, lat, lon, min_alt, ra, dec):
    if n_clicks == 0:
        return go.Figure(), go.Figure(), ""

    # Generate synthetic phase-folded data
    phase = np.linspace(-0.5, 0.5, 1000)
    flux = np.ones_like(phase)
    transit_phase = duration / 24.0 / period
    in_transit = np.abs(phase) < transit_phase / 2
    flux[in_transit] -= 0.01

    phase_fig = go.Figure()
    phase_fig.add_trace(go.Scatter(x=phase, y=flux, mode='lines', name='Model'))
    phase_fig.update_layout(
        title='Phase-Folded Transit',
        xaxis_title='Phase',
        yaxis_title='Normalized Flux',
        hovermode='x'
    )

    # Visibility plot (simplified)
    times = np.linspace(0, 24, 100)
    altitude = 30 + 40 * np.sin(2 * np.pi * times / 24)

    vis_fig = go.Figure()
    vis_fig.add_trace(go.Scatter(x=times, y=altitude, mode='lines', name='Altitude'))
    vis_fig.add_hline(y=min_alt, line_dash='dash', line_color='red',
                      annotation_text=f'Min Alt {min_alt}°')
    vis_fig.update_layout(
        title='Target Altitude Over Night',
        xaxis_title='Hour',
        yaxis_title='Altitude (°)',
        yaxis_range=[0, 90]
    )

    # Transit table
    table = html.Table([
        html.Thead(html.Tr([html.Th("Transit #"), html.Th("Time (UTC)"), html.Th("Visibility")])),
        html.Tbody([
            html.Tr([html.Td(i), html.Td(f"2024-12-{i:02d} 20:00"),
                     html.Td("✓" if i % 2 == 0 else "✗")])
            for i in range(1, 11)
        ])
    ], style={'width': '100%', 'border': '1px solid black'})

    return phase_fig, vis_fig, table

if __name__ == '__main__':
    app.run_server(debug=True, port=8050)
'''


def create_dash_app(filename="transit_dash_app.py"):
    """
    Create Dash app file.
    """
    with open(filename, 'w') as f:
        f.write(DASH_APP_CODE)

    print(f"Created Dash app: {filename}")
    print(f"Run with: python {filename}")
    print(f"Then open: http://127.0.0.1:8050")


# ============================================================
# 15. 3D ORBIT ANIMATION (MANIM + POLIASTRO)
# ============================================================

MANIM_ANIMATION_CODE = '''
"""
3D Orbit Animation using Manim and Poliastro

To render: manim -pql orbit_animation.py OrbitScene
"""

from manim import *
import numpy as np
from poliastro.bodies import Sun as PoliSun
from poliastro.twobody import Orbit
import astropy.units as u
from astropy.constants import G, M_sun

class OrbitScene(ThreeDScene):
    def construct(self):
        # Set up 3D view
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)

        # Create star (Sun)
        star = Sphere(radius=0.3, color=YELLOW)
        star_label = Text("Star", font_size=24).next_to(star, UP)

        # Define orbit parameters
        period_days = 3.5
        star_mass_solar = 1.0
        ecc = 0.3
        inc_deg = 20

        # Calculate semi-major axis using Kepler's 3rd law
        P = period_days * u.day
        M = star_mass_solar * M_sun
        a_cubed = (G * M * P**2) / (4 * np.pi**2)
        a = a_cubed ** (1 / 3)
        a_au = a.to(u.AU).value

        # Create orbit with poliastro
        orbit = Orbit.from_classical(
            PoliSun,
            a_au * u.AU,
            ecc=ecc * u.one,
            inc=inc_deg * u.deg,
            raan=0 * u.deg,
            argp=90 * u.deg,
            nu=0 * u.deg,
        )

        # Generate orbit path points
        n_points = 100
        angles = np.linspace(0, 2 * np.pi, n_points)
        orbit_points = []

        for angle in angles:
            # Ellipse parametric equations
            r = a_au * (1 - ecc**2) / (1 + ecc * np.cos(angle))
            x = r * np.cos(angle) * 2  # Scale for visibility
            y = r * np.sin(angle) * 2 * np.cos(inc_deg * np.pi / 180)
            z = r * np.sin(angle) * 2 * np.sin(inc_deg * np.pi / 180)
            orbit_points.append([x, y, z])

        # Create orbit path
        orbit_path = VMobject()
        orbit_path.set_points_as_corners(orbit_points + [orbit_points[0]])
        orbit_path.set_color(BLUE)

        # Create planet
        planet = Sphere(radius=0.15, color=BLUE)
        planet.move_to(orbit_points[0])
        planet_label = Text("Planet", font_size=20).next_to(planet, RIGHT)

        # Add everything to scene
        self.add(star, star_label)
        self.add(orbit_path)
        self.add(planet, planet_label)

        # Animate planet moving along orbit
        self.play(
            MoveAlongPath(planet, orbit_path),
            MaintainPositionRelativeTo(planet_label, planet),
            run_time=8,
            rate_func=linear
        )

        # Rotate camera for 3D effect
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(4)
        self.stop_ambient_camera_rotation()

        # Add info text
        info = VGroup(
            Text(f"Period: {period_days} days", font_size=20),
            Text(f"Semi-major axis: {a_au:.3f} AU", font_size=20),
            Text(f"Eccentricity: {ecc}", font_size=20),
            Text(f"Inclination: {inc_deg}°", font_size=20),
        ).arrange(DOWN, aligned_edge=LEFT).to_corner(UL)

        self.add_fixed_in_frame_mobjects(info)
        self.play(Write(info))
        self.wait(2)
'''


def create_manim_animation(filename="orbit_animation.py"):
    """
    Create Manim animation script.
    """
    with open(filename, 'w') as f:
        f.write(MANIM_ANIMATION_CODE)

    print(f"Created Manim animation: {filename}")
    print("To render:")
    print(f"  Low quality: manim -pql {filename} OrbitScene")
    print(f"  High quality: manim -pqh {filename} OrbitScene")


# ============================================================
# 16. EXAMPLE USAGE OF ADVANCED FEATURES
# ============================================================

def demo_advanced_features():
    """
    Demonstrate all advanced features.
    """
    print("\n" + "="*70)
    print("ADVANCED FEATURES DEMONSTRATION")
    print("="*70)

    # 1. Batman limb-darkened fitting
    print("\n[1] Limb-Darkened Transit Fitting (Batman)")
    print("Example usage:")
    print("  batman_model, params = fit_batman_transit(time, flux, period, t0)")
    print("  plot_batman_fit(time, flux, period, t0, batman_model)")

    # 2. Transmission spectroscopy
    print("\n[2] Transmission Spectroscopy Analysis")
    H = estimate_atmospheric_scale_height(
        planet_mass_earth=10,
        planet_radius_earth=3.5,
        temperature_k=1200
    )
    print(f"  Example: Hot Jupiter scale height = {H:.1f} km")

    # 3. MCMC fitting
    print("\n[3] Bayesian MCMC Fitting")
    print("Example usage:")
    print("  trace, summary = run_mcmc_transit_fit(time, flux, flux_err, period, t0)")
    print("  plot_mcmc_corner(trace)")

    # 4. Gaia lookup
    print("\n[4] Automated Gaia Lookup")
    print("Example usage:")
    print("  params = query_gaia_stellar_params(ra_deg=290, dec_deg=44.5)")
    print("  summary = create_target_summary('Kepler-9', 290, 44.5)")

    # 5. Create web apps
    print("\n[5] Creating Web Dashboard Files")
    create_streamlit_app()
    create_dash_app()

    # 6. Create animation
    print("\n[6] Creating 3D Animation Script")
    create_manim_animation()

    print("\n" + "="*70)
    print("All advanced feature templates created!")
    print("="*70)


if __name__ == "__main__":
    # Run demonstration
    demo_advanced_features()